In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import pickle
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import signal
import nrrd

In [ ]:
fname = '/work/jprieto/data/remote/EGower/jprieto/trachoma_normals_healthy_sev123_05182021_stack_32_384_test_10042021'

df = pd.read_csv(fname + '_prediction.csv')

csv_true_column = 'class'
csv_prediction_column = 'prediction'

df[csv_true_column] = (df[csv_true_column] >= 1).astype(int)
print(df[csv_prediction_column][0])
df[csv_prediction_column] = df[csv_prediction_column].astype(int)

with open(fname + '_values.pickle', 'rb') as f:
    values = pickle.load(f)

with open(fname + '_frames_prediction.pickle', 'rb') as f:
    frames_prediction = pickle.load(f)
    
with open(fname + '_frames_values.pickle', 'rb') as f:
    frames_values = pickle.load(f)

with open(fname + '_scores.pickle', 'rb') as f:
    scores = pickle.load(f)

print(df)
print(len(values), len(frames_prediction), len(frames_values), len(scores))

In [ ]:
y_true_arr = [] 
y_pred_arr = []
for idx, row in df.iterrows():
  y_true_arr.append(row[csv_true_column])
  y_pred_arr.append(row[csv_prediction_column])

In [ ]:
cnf_matrix = confusion_matrix(y_true_arr, y_pred_arr)
print(cnf_matrix)
cnf_matrix_norm = cnf_matrix.astype('float') / cnf_matrix.sum(axis=1)[:, np.newaxis]
print(cnf_matrix_norm)
print(classification_report(y_true_arr, y_pred_arr))

In [ ]:
plt.figure(figsize=[15,10])
sns.heatmap(cnf_matrix_norm, annot=True, fmt='.2%', cmap='Blues')
plt.show()

In [ ]:
values_np = np.reshape(np.array(values), (-1, 256)).astype(float)
print(np.shape(values_np), values_np.dtype)
values_embedded = TSNE(n_components=2, init='pca').fit_transform(values_np)
df["tsne_0"] = values_embedded[:,0]
df["tsne_1"] = values_embedded[:,1]
# result_df["scores_max"] = [np.max(s.reshape(-1), axis=0) for s in scores]
# result_df["prediction_abs"] = np.abs(result_df["truth"] - result_df["pred"]) = np.reshape(np.array(features), (-1, 128)).astype(float)
# print(np.shape(features_np), features_np.dtype)
# X_embedded = TSNE(n_components=2, init='pca').fit_transform(features_np)
# result_df["tsne_0"] = X_embedded[:,0]
# result_df["tsne_1"] = X_embedded[:,1]
# result_df["scores_max"] = [np.max(s.reshape(-1), axis=0) for s in scores]
# result_df["prediction_abs"] = np.abs(result_df["truth"] - result_df["pred"])

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df["tsne_0"], y=df["tsne_1"], mode='markers', text=df["prediction"], marker=dict(color=df["class"], size=(df["prediction"]+1)*5, showscale=True)))
fig.show()

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(values_np)
df["pca_0"] = X_pca[:,0]
df["pca_1"] = X_pca[:,1]

In [ ]:

fig = go.FigureWidget(make_subplots(rows=2, cols=2, column_widths=[0.7, 0.3], specs=[[{'colspan': 2}, {}],[{},{}]]))

fig.add_trace(go.Scatter(x=df["pca_0"], y=df["pca_1"], mode='markers', showlegend=False, marker=dict(
    color=df["class"], size=(df["prediction"]+1)*5, colorscale='sunset', showscale=True, opacity=1, line=dict(color='red', width=1)
)), row=1, col=1)
fig.add_trace(go.Scatter(mode='markers', showlegend=False, marker=dict(showscale=True, cmin=0, cmax=1, size=10, colorscale='sunset', line=dict(color='red', width=1))), row=2, col=1)
fig.add_trace(go.Scatter(mode='markers', marker=dict(color='LightSkyBlue', size=10), showlegend=False), row=2, col=1)
fig.add_trace(go.Image(), row=2, col=2)

# fig.data[0].marker.showscale = True
# fig.data[0].marker.colorbar.y = 0.8
# fig.data[0].marker.colorbar.len = 0.5

# fig.data[1].marker.showscale = True
# fig.data[1].marker.colorbar.y = .2
# fig.data[1].marker.colorbar.len = 0.5

fig.update_layout(
    autosize=False,
    width=1200,
    height=800
)

current_idx = {"idx": 0, "idx_f": 0, "img_np": []}

def update_study(trace, points, selector):
    if points.trace_name == 'trace 0' and len(points.point_inds) > 0:
        print("update_study", points)
        idx = points.point_inds[0]  
        x_feat_idx = np.array(frames_values[idx]).reshape(-1, 256)
        x_feat_idx_pca = pca.transform(x_feat_idx)
#         scores_idx = np.array(scores[idx]).reshape(-1)
#         weights_idx = np.array(weights[idx]).reshape(-1)
        print(frames_prediction[idx])
        df_idx = pd.DataFrame({
            "pca_0": x_feat_idx_pca[:,0],
            "pca_1": x_feat_idx_pca[:,1],
            "prediction": np.array(frames_prediction[idx]).reshape(-1)
#             ,
#             "scores": scores_idx,
#             "weights": weights_idx
            })

        with fig.batch_update():
            fig.data[1]['x'] = df_idx["pca_0"]
            fig.data[1]['y'] = df_idx["pca_1"]
            fig.data[1].marker.color = df_idx['prediction']
            fig.data[1].marker.cmin = 0
            fig.data[1].marker.cmax = 1
            print(df_idx['prediction'])

            fig.data[2]['x'] = [df.loc[idx]["pca_0"]]
            fig.data[2]['y'] = [df.loc[idx]["pca_1"]]

            current_idx["idx"] = idx
            current_idx["img_np"] = nrrd.read(df.loc[idx]["img"], index_order='C')[0]
    
fig.data[0].on_click(update_study)

def update_img(trace, points, selector):
    if points.trace_name == 'trace 1' and len(points.point_inds) > 0:
        print('update_img', points)
        idx_f = points.point_inds[0] 
        current_idx["idx_f"] = idx_f
        with fig.batch_update():
            fig.data[3]['z'] = current_idx["img_np"][idx_f]
    
fig.data[1].on_click(update_img)

fig